In [1]:
import os, json, pickle, subprocess
from google.colab import drive
drive.mount('/content/drive')
D = "/content/drive/MyDrive/"
if not os.path.exists("/content/raw"):
    subprocess.run(["unzip","-q","-o",D+"raw_videos.zip","-d","/content/"], check=True)
metadata = json.load(open(D+"metadata.json")); split = json.load(open(D+"split_v1.json"))
def vid_id(n): return n.replace(".mp4","")
label_of = {vid_id(n): (0 if i["label"]=="REAL" else 1) for n,i in metadata.items()}
print("Split counts:", {k: len(v) for k,v in split.items()})

Mounted at /content/drive
Split counts: {'train': 292, 'val': 54, 'test': 54}


In [2]:
import pandas as pd, numpy as np
def load_scores(name):
    out = {}
    for part in ["val","test"]:
        df = pd.read_csv(f"{D}{name}_{part}_video_scores.csv")
        print(name, part, df.shape, list(df.columns)); out[part] = df
    return out
vis = load_scores("visual"); fac = load_scores("facial")
print(vis["val"].head(3)); print(fac["val"].head(3))

visual val (54, 3) ['video', 'fake_prob', 'label']
visual test (54, 3) ['video', 'fake_prob', 'label']
facial val (54, 3) ['video', 'fake_prob', 'label']
facial test (54, 3) ['video', 'fake_prob', 'label']
        video  fake_prob  label
0  acxwigylke   0.998893      1
1  aelfnikyqj   0.001595      0
2  aettqgevhz   0.994475      1
        video  fake_prob  label
0  cfxkpiweqt   0.083181      0
1  bbhtdfuqxq   0.716862      1
2  cpjxareypw   0.658195      0


In [3]:
import librosa, torch
from tqdm import tqdm
from transformers import Wav2Vec2Model
SR, TARGET = 16000, 64000
pk = pickle.load(open(D+"voice_model_v2.pkl","rb"))
v_scaler, v_clf, v_layer = pk["scaler"], pk["clf"], pk["layer"]
xlsr = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-xls-r-300m").cuda().eval().half()

def clip_from_video(vid):
    raw = subprocess.run(["ffmpeg","-v","error","-i",f"/content/raw/{vid}.mp4","-vn","-t","8",
                          "-ac","1","-ar","16000","-f","f32le","pipe:1"], capture_output=True).stdout
    y_ = np.frombuffer(raw, dtype=np.float32).copy()
    if len(y_) < 1600: return None                       # no audio -> abstain
    y_, _ = librosa.effects.trim(y_, top_db=30)
    if len(y_) < 400: y_ = np.zeros(TARGET, dtype=np.float32) + 1e-4
    if len(y_) < TARGET: y_ = np.tile(y_, int(np.ceil(TARGET/len(y_))))
    y_ = y_[:TARGET].astype(np.float32)
    return (y_ - y_.mean()) / np.sqrt(y_.var() + 1e-7)   # same preprocessing as 2C

@torch.no_grad()
def voice_scores(vids, bs=8):
    out = {}
    for i in tqdm(range(0, len(vids), bs)):
        clips = [(v, clip_from_video(v)) for v in vids[i:i+bs]]
        ok = [(v, c) for v, c in clips if c is not None]
        for v, c in clips:
            if c is None: out[v] = np.nan
        if ok:
            x = torch.tensor(np.stack([c for _, c in ok])).cuda().half()
            h = xlsr(x, output_hidden_states=True).hidden_states[v_layer].float().mean(1).cpu().numpy()
            for (v, _), p in zip(ok, v_clf.predict_proba(v_scaler.transform(h))[:,1]): out[v] = float(p)
    return out

voice = {}
for part in ["val","test"]:
    voice[part] = voice_scores([vid_id(v) for v in split[part]])
print({p: (len(d), sum(np.isnan(x) for x in d.values())) for p, d in voice.items()}, "(videos, no-audio)")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.27GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.codevectors        | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.27GB            

model.safetensors: downloading bytes:           |  0.00B            



  0%|          | 0/7 [00:00<?, ?it/s]

 14%|█▍        | 1/7 [00:34<03:26, 34.46s/it]

 29%|██▊       | 2/7 [00:36<01:18, 15.66s/it]

 43%|████▎     | 3/7 [00:39<00:38,  9.61s/it]

 57%|█████▋    | 4/7 [00:41<00:19,  6.62s/it]

 71%|███████▏  | 5/7 [00:43<00:09,  4.97s/it]

 86%|████████▌ | 6/7 [00:45<00:03,  3.99s/it]

100%|██████████| 7/7 [00:13<00:00,  1.98s/it]

{'val': (54, np.int64(0)), 'test': (54, np.int64(0))} (videos, no-audio)


In [4]:
ls = pickle.load(open(D+"lipsync_model.pkl","rb")); sync_feats = pickle.load(open(D+"sync_features_v1.pkl","rb"))
def lip_score(v):
    f = sync_feats.get(v)
    if f is None or not f["applicable"]: return np.nan
    vals = {**f, "lag_abs": abs(f["lag_best"])}
    X = pd.DataFrame([[vals[k] for k in ls["feats"]]], columns=ls["feats"])
    return float(ls["clf"].predict_proba(ls["scaler"].transform(X))[0,1])
print("2D ready:", sum(not np.isnan(lip_score(vid_id(v))) for v in split["val"]), "of", len(split["val"]), "val videos scored")

2D ready: 54 of 54 val videos scored


In [5]:
def pick(df, name):
    vc = next(c for c in df.columns if "vid" in c.lower()); pc = next(c for c in df.columns if "prob" in c.lower())
    return pd.DataFrame({"vid": df[vc].astype(str).str.replace(".mp4","",regex=False), name: df[pc].values})

def build(part):
    t = pd.DataFrame({"vid": [vid_id(v) for v in split[part]]}); t["label"] = t.vid.map(label_of)
    t = t.merge(pick(vis[part],"s2A"), on="vid", how="left").merge(pick(fac[part],"s2B"), on="vid", how="left")
    t["s2C"] = t.vid.map(voice[part]); t["s2D"] = t.vid.map(lip_score)
    return t

fus = {p: build(p) for p in ["val","test"]}
for p, t in fus.items():
    t.to_csv(f"{D}fusion_table_{p}.csv", index=False)
    print(p, t.shape, "| missing scores per module:", t[["s2A","s2B","s2C","s2D"]].isna().sum().to_dict())
print(fus["val"].head())

val (54, 6) | missing scores per module: {'s2A': 0, 's2B': 0, 's2C': 0, 's2D': 0}
test (54, 6) | missing scores per module: {'s2A': 0, 's2B': 0, 's2C': 0, 's2D': 2}
          vid  label       s2A       s2B       s2C       s2D
0  cfxkpiweqt      0  0.013514  0.083181  0.846560  0.387956
1  bbhtdfuqxq      1  0.228076  0.716862  0.434652  0.419286
2  cpjxareypw      0  0.102166  0.658195  0.434652  0.395212
3  clrycekyst      0  0.006394  0.477719  0.301491  0.457909
4  alaijyygdv      1  0.990862  0.337197  0.904823  0.603699


In [6]:
from sklearn.metrics import roc_auc_score
v = fus["val"]; print("val real/fake:", int((v.label==0).sum()), int((v.label==1).sum()))
for m in ["s2A","s2B","s2C","s2D"]:
    d = v.dropna(subset=[m]); print(f"{m}: VAL AUC {roc_auc_score(d.label, d[m]):.3f}  (n={len(d)})")
print(v[["s2A","s2B","s2C","s2D"]].corr(method="spearman").round(2))

val real/fake: 12 42
s2A: VAL AUC 0.954  (n=54)
s2B: VAL AUC 0.736  (n=54)
s2C: VAL AUC 0.425  (n=54)
s2D: VAL AUC 0.669  (n=54)
      s2A   s2B   s2C   s2D
s2A  1.00  0.23 -0.12  0.26
s2B  0.23  1.00  0.14 -0.05
s2C -0.12  0.14  1.00  0.05
s2D  0.26 -0.05  0.05  1.00


In [7]:
from sklearn.metrics import roc_curve, balanced_accuracy_score, roc_auc_score

def fuse(df, w):                       # weighted average over modules that are present; abstentions are skipped
    S = np.zeros(len(df)); W = np.zeros(len(df))
    for m, wt in w.items():
        ok = df[m].notna().values
        S += np.where(ok, df[m].fillna(0).values*wt, 0); W += np.where(ok, wt, 0)
    return np.where(W > 0, S/np.maximum(W, 1e-9), np.nan)

def thr_youden(y, s):
    fpr, tpr, th = roc_curve(y, s); return float(th[np.argmax(tpr - fpr)])

v = fus["val"]; yv = v.label.values
cands = {"A: 2A alone": {"s2A":1.0},
         "B: 0.7*2A + 0.3*2B": {"s2A":0.7, "s2B":0.3},
         "(info) 0.5/0.5": {"s2A":0.5, "s2B":0.5},
         "(info) all 4 equal": {"s2A":1,"s2B":1,"s2C":1,"s2D":1}}
rows = []
for name, w in cands.items():
    s = fuse(v, w); t = thr_youden(yv, s)
    rows.append(dict(candidate=name, AUC=roc_auc_score(yv, s),
                     bal_acc_at_0_5=balanced_accuracy_score(yv, (s >= .5).astype(int)),
                     val_threshold=t, bal_acc_at_val_thr=balanced_accuracy_score(yv, (s >= t).astype(int))))
print(pd.DataFrame(rows).round(3).to_string(index=False))

auc_A, auc_B = rows[0]["AUC"], rows[1]["AUC"]
FINAL_W   = cands["B: 0.7*2A + 0.3*2B"] if auc_B >= auc_A else cands["A: 2A alone"]
THR_A     = rows[0]["val_threshold"]
THR_FINAL = rows[1]["val_threshold"] if auc_B >= auc_A else THR_A
print("\nFINAL weights:", FINAL_W, "| FINAL threshold:", round(THR_FINAL, 3), "| 2A-alone threshold:", round(THR_A, 3))

         candidate   AUC  bal_acc_at_0_5  val_threshold  bal_acc_at_val_thr
       A: 2A alone 0.954           0.946          0.228               0.958
B: 0.7*2A + 0.3*2B 0.933           0.946          0.375               0.958
    (info) 0.5/0.5 0.942           0.923          0.472               0.958
(info) all 4 equal 0.923           0.851          0.450               0.911

FINAL weights: {'s2A': 1.0} | FINAL threshold: 0.228 | 2A-alone threshold: 0.228


In [8]:
t = fus["test"]; yt = t.label.values
sA = fuse(t, {"s2A": 1.0}); sB = fuse(t, {"s2A": 0.7, "s2B": 0.3})

def report(name, s, thr):
    pred = (s >= thr).astype(int)
    print(f"{name:34s} AUC {roc_auc_score(yt, s):.3f} | bal acc {100*balanced_accuracy_score(yt, pred):.1f}% | "
          f"real ok {100*np.mean(pred[yt==0]==0):.0f}% | fake ok {100*np.mean(pred[yt==1]==1):.0f}%")

print("TEST videos:", len(yt), "| real:", int((yt==0).sum()), "| fake:", int((yt==1).sum()))
report("FINAL: 2A alone, val thr 0.228", sA, THR_A)
report("2A alone, thr 0.5 (for reference)", sA, 0.5)
report("(info) 0.7*2A+0.3*2B, val thr", sB, 0.375)

rng = np.random.default_rng(0); a = []
for _ in range(2000):
    i = rng.integers(0, len(yt), len(yt))
    if len(set(yt[i])) == 2: a.append(roc_auc_score(yt[i], sA[i]))
print("2A AUC 95%% interval: %.2f-%.2f" % tuple(np.percentile(a, [2.5, 97.5])))

TEST videos: 54 | real: 12 | fake: 42
FINAL: 2A alone, val thr 0.228     AUC 0.915 | bal acc 69.6% | real ok 42% | fake ok 98%
2A alone, thr 0.5 (for reference)  AUC 0.915 | bal acc 89.3% | real ok 83% | fake ok 95%
(info) 0.7*2A+0.3*2B, val thr      AUC 0.887 | bal acc 78.0% | real ok 58% | fake ok 98%
2A AUC 95% interval: 0.79-0.99


In [9]:
FUSION = {"weights": {"2A": 1.0, "2B": 0.0, "2C": 0.0, "2D": 0.0}, "threshold": 0.5,
          "note": "DFDC video: 2A only. 2B/2C/2D logged but weight 0 (no measurable added value on val/test). "
                  "Val-chosen threshold 0.228 failed on test (bal acc 69.6%), so the default 0.5 is used."}

def combine(outputs, cfg=FUSION):
    """outputs: {'2A': {'score': p or None, 'applicable': bool}, ...} -> authenticity result with breakdown."""
    used = {m: o["score"] for m, o in outputs.items()
            if o.get("applicable") and o.get("score") is not None and cfg["weights"].get(m, 0) > 0}
    breakdown = {m: (None if not (o.get("applicable") and o.get("score") is not None) else round(100*(1-o["score"]), 1))
                 for m, o in outputs.items()}
    if not used:
        return dict(authenticity=None, label="cannot assess", breakdown=breakdown,
                    reason="no applicable module with weight > 0 (e.g. no face found)")
    wsum = sum(cfg["weights"][m] for m in used)
    p_fake = sum(cfg["weights"][m]*s for m, s in used.items())/wsum
    return dict(authenticity=round(100*(1-p_fake), 1), label="likely fake" if p_fake >= cfg["threshold"] else "likely real",
                breakdown=breakdown, used=list(used), reason="")

# self-check on the validation table: should reproduce balanced accuracy 94.6%
v = fus["val"]; preds = []
for _, r in v.iterrows():
    res = combine({m: dict(score=(None if pd.isna(r["s"+m]) else r["s"+m]), applicable=not pd.isna(r["s"+m])) for m in ["2A","2B","2C","2D"]})
    preds.append(int(res["label"] == "likely fake"))
print("val balanced acc at 0.5: %.1f%%" % (100*balanced_accuracy_score(v.label, preds)))
print(combine({"2A": dict(score=0.9, applicable=True), "2B": dict(score=0.3, applicable=True),
               "2C": dict(score=None, applicable=False), "2D": dict(score=None, applicable=False)}))
print(combine({"2A": dict(score=None, applicable=False)}))     # no face -> cannot assess, never "real"
json.dump(dict(FUSION, val_auc_2A=0.954, test_auc_2A=0.915, test_auc_2A_ci95=[0.79, 0.99], test_bal_acc_at_0_5=0.893, n_test=54),
          open(D+"fusion_config.json", "w"), indent=2)

val balanced acc at 0.5: 94.6%
{'authenticity': 10.0, 'label': 'likely fake', 'breakdown': {'2A': 10.0, '2B': 70.0, '2C': None, '2D': None}, 'used': ['2A'], 'reason': ''}
{'authenticity': None, 'label': 'cannot assess', 'breakdown': {'2A': None}, 'reason': 'no applicable module with weight > 0 (e.g. no face found)'}
